# Assignment 5 — Naive Bayes and KNN on Spambase

**Dataset:** Spambase (UCI) — classify emails as **spam (1)** or **not spam (0)**.

**What we do:**
- Train 3 Naive Bayes variants — GaussianNB, BernoulliNB, MultinomialNB
- Train a baseline KNN classifier
- Tune KNN with GridSearchCV and RandomizedSearchCV
- Compare kd_tree vs ball_tree algorithms for KNN

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df = pd.read_csv("spambase.data", header=None)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

## 1. Quick EDA

In [2]:
print(df.head())
print(df.info())
print(df.describe())
print(df.shape)
print("rows:", df.shape[0])
print("cols:", df.shape[1])

     0     1     2    3     4     5     6     7     8     9   ...    48  \
0  0.00  0.64  0.64  0.0  0.32  0.00  0.00  0.00  0.00  0.00  ...  0.00   
1  0.21  0.28  0.50  0.0  0.14  0.28  0.21  0.07  0.00  0.94  ...  0.00   
2  0.06  0.00  0.71  0.0  1.23  0.19  0.19  0.12  0.64  0.25  ...  0.01   
3  0.00  0.00  0.00  0.0  0.63  0.00  0.31  0.63  0.31  0.63  ...  0.00   
4  0.00  0.00  0.00  0.0  0.63  0.00  0.31  0.63  0.31  0.63  ...  0.00   

      49   50     51     52     53     54   55    56  57  
0  0.000  0.0  0.778  0.000  0.000  3.756   61   278   1  
1  0.132  0.0  0.372  0.180  0.048  5.114  101  1028   1  
2  0.143  0.0  0.276  0.184  0.010  9.821  485  2259   1  
3  0.137  0.0  0.137  0.000  0.000  3.537   40   191   1  
4  0.135  0.0  0.135  0.000  0.000  3.537   40   191   1  

[5 rows x 58 columns]
<class 'pandas.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 58 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0

## 2. Split and scale

- `StandardScaler` for GaussianNB / BernoulliNB / KNN
- `MinMaxScaler` for MultinomialNB (it needs non-negative values)

In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, random_state=42, test_size=0.30)

X_validation, X_test, y_validation, y_test = train_test_split(X_temp, y_temp, random_state=42, test_size=0.30)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(X_train)
X_test_mm = scaler_mm.transform(X_test)

print("train:", X_train.shape, "test:", X_test.shape)

train: (3220, 57) test: (415, 57)


## 3. Naive Bayes models

In [4]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    accuracy_score
)

gnb, bnb, mnb = GaussianNB(), BernoulliNB(), MultinomialNB()

start = time.time()
gnb.fit(X_train_scaled, y_train)
gnbtime = time.time() - start

start = time.time()
bnb.fit(X_train_scaled, y_train)
bnbtime = time.time() - start

start = time.time()
mnb.fit(X_train_mm, y_train)
mnbtime = time.time() - start

def get_metrics(model, X_test, train_time):
    y_pred = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    return [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        tn / (tn + fp),
        train_time
    ]

metrics = ["accuracy_score", "precision_score", "recall_score", "f1_score", "Specificity", "Train_time"]
table1 = pd.DataFrame({
    "GaussianNB": get_metrics(gnb, X_test_scaled, gnbtime),
    "BernoulliNB": get_metrics(bnb, X_test_scaled, bnbtime),
    "MultinomialNB": get_metrics(mnb, X_test_mm, mnbtime)
}, index=metrics)

print("table1")
print(table1)

table1
                 GaussianNB  BernoulliNB  MultinomialNB
accuracy_score     0.826506     0.910843       0.893976
precision_score    0.718182     0.928105       0.942857
recall_score       0.940476     0.845238       0.785714
f1_score           0.814433     0.884735       0.857143
Specificity        0.748988     0.955466       0.967611
Train_time         0.007540     0.011930       0.006903


## 4. KNN baseline (k=5)

In [5]:
from sklearn.neighbors import KNeighborsClassifier

knnbaseline = KNeighborsClassifier(n_neighbors=5)

start = time.time()
knnbaseline.fit(X_train_scaled, y_train)
knntime = time.time() - start

# predict
start = time.time()
knnbaseline.predict(X_test_scaled)
knnbaseline.predict_proba(X_test_scaled)[:, 1]
knnpretime = time.time() - start

knn_baseline_metric = get_metrics(knnbaseline, X_test_scaled, knntime)

print("knnbaseline prediction metrics")
print("accuracy_score:", knn_baseline_metric[0])
print("precision_score:", knn_baseline_metric[1])
print("recall_score:", knn_baseline_metric[2])
print("f1_score:", knn_baseline_metric[3])
print("Specificity:", knn_baseline_metric[4])
print("Train_time:", knn_baseline_metric[5])
print("knnpretime:", knnpretime)

knnbaseline prediction metrics
accuracy_score: 0.8987951807228916
precision_score: 0.8841463414634146
recall_score: 0.8630952380952381
f1_score: 0.8734939759036144
Specificity: 0.9230769230769231
Train_time: 0.0017895698547363281
knnpretime: 2.697617292404175


## 5. KNN tuning — GridSearchCV vs RandomizedSearchCV

- `p=1` is manhattan distance, `p=2` is euclidean distance

In [6]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

parameter = {
    "n_neighbors": list(range(1, 21)),
    "weights": ["uniform", "distance"],
    "p": [1, 2]  # 1 manhattan distance and 2 euclidean distance
}

gridknn = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=parameter,
    cv=5,
    scoring="accuracy",
    n_jobs=1
)

start = time.time()
gridknn.fit(X_train_scaled, y_train)
gridknntime = time.time() - start

randomknn = RandomizedSearchCV(
    estimator=KNeighborsClassifier(),
    param_distributions=parameter,
    scoring="accuracy",
    n_iter=10,
    n_jobs=1
)

start = time.time()
randomknn.fit(X_train_scaled, y_train)
randomknntime = time.time() - start

table2 = pd.DataFrame({
    "SearchMethod": ["GridSearch", "RandomizedSearch"],
    "Best K": [gridknn.best_params_["n_neighbors"], randomknn.best_params_["n_neighbors"]],
    "Best Cv accuracy": [gridknn.best_score_, randomknn.best_score_],
    "Best Parameter": [str(gridknn.best_params_), str(randomknn.best_params_)]
})
print("knn classifier in the gridsearch and randomized search")
print(table2.to_string(index=False))
print("grid search time:", gridknntime)
print("random search time:", randomknntime)

knn classifier in the gridsearch and randomized search
    SearchMethod  Best K  Best Cv accuracy                                     Best Parameter
      GridSearch      10          0.922671 {'n_neighbors': 10, 'p': 1, 'weights': 'distance'}
RandomizedSearch       6          0.920186  {'weights': 'distance', 'p': 1, 'n_neighbors': 6}
grid search time: 10.678146600723267
random search time: 1.4985222816467285


## 6. KNN with kd_tree and ball_tree

Using the best params found by grid search.

In [7]:
bestk = gridknn.best_params_["n_neighbors"]
best_weight = gridknn.best_params_["weights"]
best_p = gridknn.best_params_["p"]

knnkdtree = KNeighborsClassifier(
    n_neighbors=bestk,
    weights=best_weight,
    p=best_p,
    algorithm='kd_tree'
)

knnkdtree.fit(X_train_scaled, y_train)

start = time.time()
knnkdtree.predict(X_test_scaled)
knnkdtreetime = time.time() - start

knnkdtreemetric = get_metrics(knnkdtree, X_test_scaled, knnkdtreetime)

knnballtree = KNeighborsClassifier(
    n_neighbors=bestk,
    weights=best_weight,
    p=best_p,
    algorithm="ball_tree"
)

knnballtree.fit(X_train_scaled, y_train)

start = time.time()
knnballtree.predict(X_test_scaled)
knnballtreetime = time.time() - start

knnballtreemetric = get_metrics(knnballtree, X_test_scaled, knnballtreetime)

table3 = pd.DataFrame({
    "metric": ["accuracy_score", "precision_score", "recall_score", "f1_score", "Specificity", "Train_time"],
    "result of kd tree": [knnkdtreemetric[0], knnkdtreemetric[1], knnkdtreemetric[2], knnkdtreemetric[3], knnkdtreemetric[4], knnkdtreemetric[5]],
    "result of ball tree": [knnballtreemetric[0], knnballtreemetric[1], knnballtreemetric[2], knnballtreemetric[3], knnballtreemetric[4], knnballtreemetric[5]]
})

print(table3)

            metric  result of kd tree  result of ball tree
0   accuracy_score           0.944578             0.944578
1  precision_score           0.973856             0.973856
2     recall_score           0.886905             0.886905
3         f1_score           0.928349             0.928349
4      Specificity           0.983806             0.983806
5       Train_time           0.130083             0.085966


## 7. Final comparison — all models

In [8]:
# time the kd_tree / ball_tree fits too so the table is complete
start = time.time()
knnkdtree.fit(X_train_scaled, y_train)
kdtree_train_time = time.time() - start

start = time.time()
knnballtree.fit(X_train_scaled, y_train)
balltree_train_time = time.time() - start

def full_metrics(model, X_te, train_time):
    start = time.time()
    y_pred = model.predict(X_te)
    pred_time = time.time() - start
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    return [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        tn / (tn + fp),
        train_time,
        pred_time
    ]

all_models = [
    ("GaussianNB",          gnb,                        X_test_scaled, gnbtime),
    ("BernoulliNB",         bnb,                        X_test_scaled, bnbtime),
    ("MultinomialNB",       mnb,                        X_test_mm,     mnbtime),
    ("KNN baseline (k=5)",  knnbaseline,                X_test_scaled, knntime),
    ("KNN GridSearch",      gridknn.best_estimator_,    X_test_scaled, gridknntime),
    ("KNN RandomizedSearch",randomknn.best_estimator_,  X_test_scaled, randomknntime),
    ("KNN kd_tree",         knnkdtree,                  X_test_scaled, kdtree_train_time),
    ("KNN ball_tree",       knnballtree,                X_test_scaled, balltree_train_time),
]

cols = ["accuracy", "precision", "recall", "f1_score", "specificity", "train_time_s", "predict_time_s"]
comparison = pd.DataFrame(
    {name: full_metrics(m, Xte, tt) for name, m, Xte, tt in all_models},
    index=cols
).T

print(comparison.round(4).to_string())

                      accuracy  precision  recall  f1_score  specificity  train_time_s  predict_time_s
GaussianNB              0.8265     0.7182  0.9405    0.8144       0.7490        0.0075          0.0009
BernoulliNB             0.9108     0.9281  0.8452    0.8847       0.9555        0.0119          0.0009
MultinomialNB           0.8940     0.9429  0.7857    0.8571       0.9676        0.0069          0.0004
KNN baseline (k=5)      0.8988     0.8841  0.8631    0.8735       0.9231        0.0018          0.0126
KNN GridSearch          0.9446     0.9739  0.8869    0.9283       0.9838       10.6781          0.0259
KNN RandomizedSearch    0.9253     0.9255  0.8869    0.9058       0.9514        1.4985          0.0229
KNN kd_tree             0.9446     0.9739  0.8869    0.9283       0.9838        0.0451          0.1105
KNN ball_tree           0.9446     0.9739  0.8869    0.9283       0.9838        0.0249          0.0715


## 8. Conclusion

- MultinomialNB / BernoulliNB suit count or binary features; GaussianNB handles continuous ones.
- KNN baseline already beats the NB models on accuracy.
- Grid search checks every combo, random search samples — both land on similar best params.
- kd_tree and ball_tree give the same predictions; they only differ in how fast they find neighbours.